# GeoNUSAF — Recovering the tile ground sampling distance from photogrammetric control

**Calibration tile: `image264.png` (Dasharath Rangasala, Kathmandu)**

Same 20-section layout as the baseline notebooks: `@dataclass CFG`, markdown-then-code alternation,
asserts at every point where a wrong answer could pass silently. Nothing here trains and nothing
writes to `DATA_ROOT`.

## Why this notebook exists

Every tile in the dataset is a 2345 x 2351 px raster exported from a QGIS print-layout Atlas over
the Google Satellite basemap. The Atlas project was lost, so the **ground extent of one tile is not
recorded anywhere in the data**. Without it the dataset can only be described in pixels: no ground
sampling distance, no study-area size in km², no road widths in metres, no patch areas in hectares.
The working assumption up to now — a 300 m tile side — was a guess with nothing behind it.

Tile geometry cannot be recovered from the files themselves. It can be recovered from **the ground**,
because some objects in the scene have dimensions fixed by international regulation. The football
pitch inside Dasharath Rangasala carries four such objects. Under the IFAB Laws of the Game the
penalty area is exactly 40.32 m x 16.5 m and the goal area exactly 18.32 m x 5.5 m, **regardless of
the overall pitch size**, which itself is allowed to vary. Those four distances are rulers already
lying on the ground inside the imagery.

The method:

1. Locate the pitch automatically by colour and shape — no hand-drawn crop.
2. Rotate it upright using the fitted rectangle's own angle.
3. Enhance the painted white lines by subtracting a local median background.
4. Find the lines as peaks in 1-D intensity profiles, refined to sub-pixel by intensity centroid.
5. Convert each of the four regulation distances into an independent GSD estimate.
6. Validate against three things the method did not use: the halfway-line symmetry, the overall
   pitch dimensions against the regulation envelope, the athletics track, and a 50 m pool in a
   different tile.
7. Rescale every metre-denominated dataset statistic and emit the corrected table.

## What it does not claim

This recovers **scale**, not position. Tile coordinates, the Atlas grid and the tile ordering remain
unknown; nothing here makes DEM conditioning or true spatial cross-validation possible.

## Cost

Two tiles are decoded. The whole notebook runs in under a minute on CPU. No GPU, no internet.

## 1. Environment

Nothing is installed. Versions are printed rather than pinned so the exact values can be quoted in
the reproducibility paragraph. `cv2.setNumThreads(0)` is kept for consistency with the other
notebooks; it has no material effect at this scale.

In [ ]:
import os, json, math, warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import cv2
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)
matplotlib.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9})

print("numpy", np.__version__, "| opencv", cv2.__version__,
      "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__)

## 2. Configuration

One dataclass, as in the baselines. The values that matter:

- `CALIB_TILE` — the tile carrying the football pitch. This is the only tile the measurement uses.
- `POOL_TILE` — a second tile used purely as an independent cross-check in section 15.
- `LAWS` — the IFAB Laws of the Game constants. These are the rulers. Every one of them is a fixed
  number, not a typical value, and they have not changed since long before the imagery was captured.
- `OLD_TILE_SIDE_M` — the unverified 300 m assumption that the earlier characterisation used. It
  appears here only so section 17 can compute the correction factor; nothing in the measurement
  depends on it.

`SEARCH` holds the segmentation and peak-detection parameters. They are exposed rather than buried
so the result can be checked for sensitivity to them.

In [ ]:
@dataclass
class CFG:
    # ---- data ----
    DATA_ROOT: str = "/kaggle/input/datasets/sugam24/satellite-datasets/satellite_dataset"
    CALIB_TILE: str = "image264.png"      # Dasharath Rangasala — the football pitch
    POOL_TILE: str = "image1.png"         # independent cross-check (50 m pool)
    NAT_W: int = 2345
    NAT_H: int = 2351
    IMG_SIZE: int = 512                   # working resolution of every part-1 model

    # ---- IFAB Laws of the Game, Law 1 (metres). FIXED regardless of pitch size ----
    PENALTY_AREA_W: float = 40.32         # 44 yd
    PENALTY_AREA_D: float = 16.50         # 18 yd
    GOAL_AREA_W: float = 18.32            # 20 yd
    GOAL_AREA_D: float = 5.50             # 6 yd
    CENTRE_CIRCLE_R: float = 9.15         # 10 yd
    PITCH_LEN_MIN: float = 100.0          # regulation envelope, for validation only
    PITCH_LEN_MAX: float = 110.0
    PITCH_WID_MIN: float = 64.0
    PITCH_WID_MAX: float = 75.0

    # ---- other control objects, for cross-checks only ----
    TRACK_OUTER_LEN: float = 176.91       # 8-lane 400 m track, outer edge
    TRACK_OUTER_WID: float = 92.50
    POOL_LEN: float = 50.0                # long-course pool

    # ---- the assumption being replaced ----
    OLD_TILE_SIDE_M: float = 300.0

    # ---- segmentation / detection parameters ----
    GREEN_H: Tuple[int, int] = (35, 95)
    GREEN_S_MIN: int = 40
    GREEN_V_MIN: int = 40
    CLOSE_K: int = 11
    MIN_BLOB_AREA: int = 20000
    ASPECT_RANGE: Tuple[float, float] = (1.30, 1.80)
    MIN_FILL: float = 0.85
    PAD: int = 12                         # border kept around the rotated pitch, px
    MEDIAN_BG: int = 21                   # local-median window for white-line enhancement
    PROFILE_BAND: Tuple[float, float] = (0.23, 0.76)   # central band for the longitudinal profile
    PEAK_K_LONG: float = 1.0              # threshold = mean + k*std
    PEAK_K_TRANS: float = 1.2
    PEAK_MIN_SEP: int = 5
    SUBPX_R: int = 3                      # +/- window for the intensity-centroid refinement
    SYMMETRY_TOL: float = 5.0             # px, for the symmetric-pair search

    # ---- output ----
    OUT_DIR: str = "/kaggle/working/gsd_calibration"
    FIG_DIR: str = "/kaggle/working/gsd_calibration/figs"

cfg = CFG()
os.makedirs(cfg.OUT_DIR, exist_ok=True)
os.makedirs(cfg.FIG_DIR, exist_ok=True)

RESULTS: Dict[str, object] = {}
LATEX: Dict[str, str] = {}

print("DATA_ROOT   :", cfg.DATA_ROOT)
print("calibration :", cfg.CALIB_TILE)
print("cross-check :", cfg.POOL_TILE)
print("outputs     :", cfg.OUT_DIR)

## 3. Load the calibration tile

The tile is read straight from the attached Kaggle dataset — not from a re-saved copy — so the
pixel grid is exactly the one every other notebook uses. The raster size is asserted against the
dataset-wide value verified in the characterisation notebook, because the entire calibration is a
statement about *this* pixel grid: a tile of a different size would have a different GSD.

An MD5 of the file is printed so the exact input can be cited in the paper and re-checked later.

In [ ]:
import hashlib

IMG_DIR = Path(cfg.DATA_ROOT) / "images"
calib_path = IMG_DIR / cfg.CALIB_TILE
assert calib_path.exists(), f"not found: {calib_path}"

img = cv2.imread(str(calib_path), cv2.IMREAD_COLOR)
assert img is not None, "decode failed"
H_PX, W_PX = img.shape[:2]
print(f"{cfg.CALIB_TILE}: {W_PX} x {H_PX} px, {calib_path.stat().st_size/1e6:.2f} MB")
print("md5:", hashlib.md5(calib_path.read_bytes()).hexdigest())

assert (W_PX, H_PX) == (cfg.NAT_W, cfg.NAT_H), \
    f"raster is {W_PX}x{H_PX}, expected {cfg.NAT_W}x{cfg.NAT_H} — the GSD would not transfer"

RESULTS["input"] = {"tile": cfg.CALIB_TILE, "width_px": W_PX, "height_px": H_PX,
                    "md5": hashlib.md5(calib_path.read_bytes()).hexdigest()}

## 4. The control objects

| Feature | Dimension | Source |
|---|---|---|
| Penalty area | 40.32 m x 16.50 m | IFAB Laws of the Game, Law 1 |
| Goal area | 18.32 m x 5.50 m | IFAB Laws of the Game, Law 1 |
| Centre circle | 9.15 m radius | IFAB Laws of the Game, Law 1 |
| Pitch | 100-110 m x 64-75 m | Law 1 — a *range*, so validation only |

The distinction in the last row is the whole reason this works. Pitch length and width are free
within a range and therefore useless as rulers; the penalty and goal areas are fixed, so a stadium
that follows the Laws at all follows them exactly. Two of these are long baselines (40.32 m and
16.50 m across tens of pixels) and two are short, which matters for the error budget in section 12:
a fixed sub-pixel error in locating a line is a larger *relative* error on a shorter baseline.

The lines themselves are painted white on grass, roughly 12 cm wide, which is well under one pixel
here. They survive as a brightness excess spread over two or three pixels — which is precisely what
the sub-pixel centroid in section 8 is designed to recover.

In [ ]:
CONTROLS = pd.DataFrame([
    {"feature": "Penalty area width",  "metres": cfg.PENALTY_AREA_W, "kind": "fixed"},
    {"feature": "Penalty area depth",  "metres": cfg.PENALTY_AREA_D, "kind": "fixed"},
    {"feature": "Goal area width",     "metres": cfg.GOAL_AREA_W,    "kind": "fixed"},
    {"feature": "Goal area depth",     "metres": cfg.GOAL_AREA_D,    "kind": "fixed"},
    {"feature": "Pitch length",        "metres": float("nan"),       "kind": "range 100-110"},
    {"feature": "Pitch width",         "metres": float("nan"),       "kind": "range 64-75"},
])
print(CONTROLS.to_string(index=False))

## 5. Locating the pitch automatically

No hand-drawn crop. The pitch is found by a rule that can be restated in one sentence in the paper:
*the largest green connected component whose minimum-area rectangle is at least 85 % filled and has
an aspect ratio between 1.30 and 1.80.*

Each clause earns its place. **Green** removes the built-up majority of the tile. **Largest** and the
20 000 px floor remove hedges and gardens. **Fill ratio** is what separates a pitch from an irregular
park or a tree canopy: a marked-out football pitch is a rectangle, so its contour fills its own
bounding rectangle almost completely, while vegetation does not. **Aspect** encodes Law 1's envelope
(100-110 by 64-75 gives 1.33 to 1.72) with a small margin.

The candidate table is printed in full so the selection can be audited — the point is that exactly
one blob in this tile satisfies the rule, and the runners-up fail it clearly rather than narrowly.

In [ ]:
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
Hc, Sc, Vc = hsv[..., 0].astype(int), hsv[..., 1].astype(int), hsv[..., 2].astype(int)
green = ((Hc > cfg.GREEN_H[0]) & (Hc < cfg.GREEN_H[1]) &
         (Sc > cfg.GREEN_S_MIN) & (Vc > cfg.GREEN_V_MIN)).astype(np.uint8)
green = cv2.morphologyEx(green, cv2.MORPH_CLOSE, np.ones((cfg.CLOSE_K, cfg.CLOSE_K), np.uint8))

n, lab, st, cen = cv2.connectedComponentsWithStats(green, 8)
rows = []
for i in range(1, n):
    if st[i, 4] < cfg.MIN_BLOB_AREA:
        continue
    m = (lab == i).astype(np.uint8)
    c = max(cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[0], key=cv2.contourArea)
    (cx, cy), (a, b), ang = cv2.minAreaRect(c)
    if a < b:
        a, b, ang = b, a, ang + 90
    rows.append({"blob": i, "area_px": int(st[i, 4]), "cx": cx, "cy": cy,
                 "len_px": a, "wid_px": b, "angle": ang,
                 "aspect": a / b, "fill": cv2.contourArea(c) / (a * b)})
cand = pd.DataFrame(rows).sort_values("area_px", ascending=False)
print(cand.to_string(index=False, float_format=lambda v: f"{v:9.3f}"))

ok = cand[(cand.aspect >= cfg.ASPECT_RANGE[0]) & (cand.aspect <= cfg.ASPECT_RANGE[1]) &
          (cand.fill >= cfg.MIN_FILL)]
assert len(ok) >= 1, "no blob satisfies the pitch rule"
pitch = ok.iloc[0]
print(f"\nselected blob {int(pitch.blob)}: {pitch.len_px:.2f} x {pitch.wid_px:.2f} px, "
      f"angle {pitch.angle:.2f} deg, fill {pitch.fill:.3f}, centre "
      f"({pitch.cx:.1f}, {pitch.cy:.1f})")
print(f"rejected {len(cand)-len(ok)} of {len(cand)} candidates on aspect/fill")

RESULTS["pitch_rect"] = {"len_px": float(pitch.len_px), "wid_px": float(pitch.wid_px),
                         "angle_deg": float(pitch.angle), "fill": float(pitch.fill),
                         "centre_px": [float(pitch.cx), float(pitch.cy)],
                         "n_candidates": int(len(cand))}

## 6. Rotating the pitch upright

The pitch sits at about 74 degrees in the tile. Measuring line separations on a rotated raster means
measuring along diagonals, so the image is rotated by the fitted rectangle's own angle — a value the
segmentation produced, not one chosen by hand — and everything after this is an axis-aligned
distance.

`INTER_CUBIC` is used because the next step reads sub-pixel intensity structure; nearest-neighbour
would quantise exactly the signal being measured. The rotation is a rigid transform, so it changes
no distances: scale is preserved by construction, which is the property that lets the measurement
survive the resampling.

A margin of `PAD` px is kept beyond the rectangle so lines lying on the pitch boundary are not
clipped by the crop.

In [ ]:
cx, cy = float(pitch.cx), float(pitch.cy)
L, W, ANG = float(pitch.len_px), float(pitch.wid_px), float(pitch.angle)

M = cv2.getRotationMatrix2D((cx, cy), ANG, 1.0)
rot = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]), flags=cv2.INTER_CUBIC)

x0, x1 = int(cx - L / 2) - cfg.PAD, int(cx + L / 2) + cfg.PAD
y0, y1 = int(cy - W / 2) - cfg.PAD, int(cy + W / 2) + cfg.PAD
P = rot[y0:y1, x0:x1]
PH, PW = P.shape[:2]
CENTRE_Y = cfg.PAD + W / 2.0
print(f"rotated crop {PW} x {PH} px | pitch spans x {cfg.PAD:.1f}..{cfg.PAD+L:.1f}, "
      f"y {cfg.PAD:.1f}..{cfg.PAD+W:.1f}")
print(f"pitch centre in crop: y = {CENTRE_Y:.2f}")

## 7. Enhancing the painted lines

The grass is not uniform — mowing stripes, wear, shadow from the stands — so a global brightness
threshold finds the mowing pattern rather than the markings. Subtracting a **local median** removes
anything that varies slowly across the pitch and keeps only structures thinner than the median
window, which is exactly the class of object a painted line belongs to.

The window is 21 px. Wide enough that a 2-3 px line cannot influence its own background estimate;
narrow enough that the background still tracks the mowing stripes.

In [ ]:
gray = cv2.cvtColor(P, cv2.COLOR_BGR2GRAY)
bg = cv2.medianBlur(gray, cfg.MEDIAN_BG)
white = np.clip(gray.astype(np.float32) - bg.astype(np.float32), 0, None)
print(f"white-line map: max {white.max():.1f}, mean {white.mean():.3f}, "
      f"99.9th pct {np.percentile(white, 99.9):.1f}")

## 8. The longitudinal profile — goal, goal-area, penalty and halfway lines

Averaging the white-line map down the columns turns every line running **across** the pitch into a
peak in a 1-D signal. The average is taken over the central 23-76 % of the pitch width, which
excludes the touchlines and the box side lines so that only transverse lines contribute.

Each peak is then refined to sub-pixel by taking the intensity centroid over +/-3 px. This is the
step that makes short baselines usable at all: a peak quantised to the nearest integer pixel would
carry +/-0.5 px of error, which on the 17.8 px goal-area depth is 2.8 %.

Seven peaks are expected and found, symmetric about the halfway line: goal line, goal-area line and
penalty-area line at each end, plus the halfway line itself.

In [ ]:
def find_peaks(p: np.ndarray, k: float, min_sep: int) -> List[Tuple[int, float]]:
    thr = p.mean() + k * p.std()
    out: List[Tuple[int, float]] = []
    for i in range(1, len(p) - 1):
        if p[i] > thr and p[i] >= p[i - 1] and p[i] >= p[i + 1]:
            if not out or i - out[-1][0] > min_sep:
                out.append((i, float(p[i])))
            elif p[i] > out[-1][1]:
                out[-1] = (i, float(p[i]))
    return out

def sub_px(p: np.ndarray, i: int, r: int) -> float:
    lo, hi = max(0, i - r), min(len(p), i + r + 1)
    seg, idx = p[lo:hi], np.arange(lo, hi)
    return float((seg * idx).sum() / seg.sum())

lo_b, hi_b = int(cfg.PROFILE_BAND[0] * PH), int(cfg.PROFILE_BAND[1] * PH)
col_prof = white[lo_b:hi_b].mean(0)
col_pk = find_peaks(col_prof, cfg.PEAK_K_LONG, cfg.PEAK_MIN_SEP)
col_x = [sub_px(col_prof, i, cfg.SUBPX_R) for i, _ in col_pk]
print("column peaks (sub-pixel x):", [round(v, 2) for v in col_x])

inside = [x for x in col_x if cfg.PAD - 3 <= x <= cfg.PAD + L + 3]
assert len(inside) >= 7, f"expected >=7 transverse lines, found {len(inside)}"
gl_L, ga_L, pa_L = inside[0], inside[1], inside[2]
gl_R, ga_R, pa_R = inside[-1], inside[-2], inside[-3]
half = min(inside[3:-3], key=lambda v: abs(v - (gl_L + gl_R) / 2))

print(f"\n  left  goal line        x = {gl_L:8.2f}")
print(f"  left  goal-area line   x = {ga_L:8.2f}")
print(f"  left  penalty line     x = {pa_L:8.2f}")
print(f"  halfway line           x = {half:8.2f}")
print(f"  right penalty line     x = {pa_R:8.2f}")
print(f"  right goal-area line   x = {ga_R:8.2f}")
print(f"  right goal line        x = {gl_R:8.2f}")

## 9. Symmetry validation

A football pitch is symmetric about the halfway line by construction, and that symmetry was **not
used** to find any of the lines. So it is a free test of the whole chain — segmentation, rotation
angle, resampling, background subtraction, peak refinement.

The halfway line must fall at the midpoint of the two goal lines. If the rotation angle were wrong,
or the resampling were shearing the raster, or the peak refinement were biased, this check would
fail. It is the closest thing available to an external check without ground truth.

The two penalty depths and the two goal-area depths are also compared left against right. These are
allowed to disagree slightly — the goal line sits at the very edge of the mown area where the
green/track boundary interferes — and section 12 turns that disagreement into the error bar rather
than hiding it.

In [ ]:
pred_half = (gl_L + gl_R) / 2
err_half = abs(half - pred_half)
pitch_len_px = gl_R - gl_L
print(f"halfway line: measured {half:.2f}, predicted {pred_half:.2f}, "
      f"error {err_half:.2f} px ({err_half/pitch_len_px*100:.2f}% of pitch length)")
assert err_half < 3.0, "halfway line is not where symmetry says — the geometry chain is broken"

d_pa_L, d_pa_R = pa_L - gl_L, gl_R - pa_R
d_ga_L, d_ga_R = ga_L - gl_L, gl_R - ga_R
print(f"\npenalty depth   : left {d_pa_L:6.2f} px | right {d_pa_R:6.2f} px | "
      f"disagreement {abs(d_pa_L-d_pa_R)/np.mean([d_pa_L,d_pa_R])*100:.1f}%")
print(f"goal-area depth : left {d_ga_L:6.2f} px | right {d_ga_R:6.2f} px | "
      f"disagreement {abs(d_ga_L-d_ga_R)/np.mean([d_ga_L,d_ga_R])*100:.1f}%")

RESULTS["longitudinal"] = {"goal_L": gl_L, "goal_R": gl_R, "goalarea_L": ga_L, "goalarea_R": ga_R,
                           "penalty_L": pa_L, "penalty_R": pa_R, "halfway": half,
                           "halfway_pred": pred_half, "halfway_err_px": err_half,
                           "pitch_len_px": pitch_len_px}

## 10. The transverse profiles — box side lines

The penalty and goal areas are bounded on their long sides by lines running **along** the pitch.
Those only exist between the goal line and the box front line, so the row profile is taken over
exactly that band — a range derived from the peaks found in section 8, not typed in.

Identification uses symmetry again: both boxes are centred on the pitch's own centre line, so the
correct pair is the one whose midpoint lands within 5 px of `CENTRE_Y`. That rules out the
touchlines, the goal frame and any stray peak without needing to know in advance how far apart the
lines should be — which matters, because the separation is precisely the unknown being measured.

In [ ]:
def transverse(x_lo: float, x_hi: float):
    prof = white[:, int(round(x_lo)):int(round(x_hi))].mean(1)
    pk = find_peaks(prof, cfg.PEAK_K_TRANS, cfg.PEAK_MIN_SEP)
    ys = [sub_px(prof, i, cfg.SUBPX_R) for i, _ in pk]
    pairs = []
    for a in range(len(ys)):
        for b in range(a + 1, len(ys)):
            mid = (ys[a] + ys[b]) / 2
            if abs(mid - CENTRE_Y) < cfg.SYMMETRY_TOL:
                pairs.append((ys[a], ys[b], ys[b] - ys[a], mid))
    return prof, ys, pairs

bands = {
    "left penalty":  (gl_L + 4, pa_L - 3),
    "right penalty": (pa_R + 3, gl_R - 4),
    "left goal-area": (gl_L + 3, ga_L - 2),
}
found = {}
for name, (a, b) in bands.items():
    prof, ys, pairs = transverse(a, b)
    print(f"\n{name} band x {a:.1f}..{b:.1f}")
    print("  peaks:", [round(v, 2) for v in ys])
    for p in pairs:
        print(f"    symmetric pair {p[0]:7.2f} / {p[1]:7.2f}  mid {p[3]:6.2f}  sep {p[2]:7.2f} px")
    found[name] = pairs

# the penalty side lines are the widest symmetric pair that is not the touchline pair
def pick(pairs, lo, hi):
    c = [p for p in pairs if lo <= p[2] <= hi]
    assert c, f"no symmetric pair with separation in [{lo}, {hi}]"
    return max(c, key=lambda p: p[2])

touch = pick(found["left penalty"], 0.85 * W, 1.02 * W)
pa_w_L = pick(found["left penalty"],  0.40 * W, 0.70 * W)[2]
pa_w_R = pick(found["right penalty"], 0.40 * W, 0.70 * W)[2]
ga_w   = pick(found["left goal-area"], 0.15 * W, 0.35 * W)[2]

print(f"\npenalty-area width : left {pa_w_L:.2f} px | right {pa_w_R:.2f} px | "
      f"disagreement {abs(pa_w_L-pa_w_R)/np.mean([pa_w_L,pa_w_R])*100:.2f}%")
print(f"goal-area width    : {ga_w:.2f} px")
print(f"touchline separation (pitch width): {touch[2]:.2f} px")

RESULTS["transverse"] = {"penalty_w_L": pa_w_L, "penalty_w_R": pa_w_R,
                         "goalarea_w": ga_w, "pitch_wid_px": touch[2]}

## 11. Four independent GSD estimates

Each regulation distance divided by its measured pixel separation is one estimate of the ground
sampling distance. They are independent in the sense that matters: different lines, different
orientations, different parts of the pitch.

Left/right pairs are averaged before dividing, since their disagreement is a property of the
measurement rather than of the ground.

In [ ]:
d_pa = (d_pa_L + d_pa_R) / 2
d_ga = (d_ga_L + d_ga_R) / 2
w_pa = (pa_w_L + pa_w_R) / 2

est = pd.DataFrame([
    {"control": "Penalty area width", "true_m": cfg.PENALTY_AREA_W, "px": w_pa},
    {"control": "Penalty area depth", "true_m": cfg.PENALTY_AREA_D, "px": d_pa},
    {"control": "Goal area width",    "true_m": cfg.GOAL_AREA_W,    "px": ga_w},
    {"control": "Goal area depth",    "true_m": cfg.GOAL_AREA_D,    "px": d_ga},
])
est["gsd_m_per_px"] = est.true_m / est.px
est["tile_side_m"] = est.gsd_m_per_px * cfg.NAT_W
print(est.to_string(index=False, float_format=lambda v: f"{v:11.4f}"))

# length-weighted combination: weighting each estimate by its baseline in pixels reduces to
# (total known metres) / (total measured pixels), which is the least-squares scale estimator
# under a constant per-line localisation error.
GSD = float(est.true_m.sum() / est.px.sum())
GSD_MED = float(est.gsd_m_per_px.median())
print(f"\nlength-weighted GSD : {GSD:.4f} m/px")
print(f"median GSD          : {GSD_MED:.4f} m/px")
print(f"range               : {est.gsd_m_per_px.min():.4f} - {est.gsd_m_per_px.max():.4f} m/px")

RESULTS["estimates"] = est.to_dict("records")
RESULTS["gsd_weighted"] = GSD
RESULTS["gsd_median"] = GSD_MED

## 12. Error budget

The four estimates do not agree perfectly, and the pattern of the disagreement is informative rather
than alarming: the two **long** baselines cluster tightly, and the two **short** ones scatter. That
is the signature of a roughly constant per-line localisation error, which is what a sub-pixel
centroid on a 2-3 px line should produce.

The implied localisation error is recovered below by asking, for each control, how many pixels the
measurement would have to move to land exactly on the weighted estimate. If those implied errors are
all of the same small size, the error model holds and the scatter is explained. If one of them were
large, that control would be measuring something other than what it claims.

The reported uncertainty is the half-range across the four controls, which is deliberately
conservative — it assumes no cancellation at all between independent measurements.

In [ ]:
est["implied_px_err"] = est.true_m / GSD - est.px
est["rel_dev_pct"] = (est.gsd_m_per_px - GSD) / GSD * 100
print(est[["control", "px", "gsd_m_per_px", "implied_px_err", "rel_dev_pct"]].to_string(
    index=False, float_format=lambda v: f"{v:11.4f}"))

half_range = (est.gsd_m_per_px.max() - est.gsd_m_per_px.min()) / 2
rel = half_range / GSD * 100
print(f"\nmean |implied localisation error| : {est.implied_px_err.abs().mean():.2f} px")
print(f"max  |implied localisation error| : {est.implied_px_err.abs().max():.2f} px")
print(f"\nGSD        = {GSD:.4f} +/- {half_range:.4f} m/px  ({rel:.1f}%)")
print(f"tile side  = {GSD*cfg.NAT_W:.1f} +/- {half_range*cfg.NAT_W:.1f} m (X), "
      f"{GSD*cfg.NAT_H:.1f} m (Y)")

assert est.implied_px_err.abs().max() < 3.0, \
    "one control disagrees by more than 3 px — it is not measuring what it claims"

RESULTS["uncertainty"] = {"half_range_m_per_px": float(half_range), "rel_pct": float(rel),
                          "max_implied_px_err": float(est.implied_px_err.abs().max())}

## 13. Cross-check 1 — the pitch itself

The overall pitch dimensions were not used to derive anything: pitch size is a *range* in Law 1, so
it cannot calibrate. But it can falsify. Converted at the recovered GSD, the pitch must land inside
100-110 m by 64-75 m. If the scale were wrong by the factor of two that a mis-identified control
would produce, the pitch would come out at 50 m or 200 m long and the error would be obvious.

In [ ]:
pl = pitch_len_px * GSD
pw = touch[2] * GSD
print(f"pitch length : {pitch_len_px:7.2f} px -> {pl:7.2f} m  "
      f"(Law 1 allows {cfg.PITCH_LEN_MIN:.0f}-{cfg.PITCH_LEN_MAX:.0f})")
print(f"pitch width  : {touch[2]:7.2f} px -> {pw:7.2f} m  "
      f"(Law 1 allows {cfg.PITCH_WID_MIN:.0f}-{cfg.PITCH_WID_MAX:.0f})")
ok_len = cfg.PITCH_LEN_MIN <= pl <= cfg.PITCH_LEN_MAX
ok_wid = cfg.PITCH_WID_MIN <= pw <= cfg.PITCH_WID_MAX
print(f"\nlength within envelope: {ok_len} | width within envelope: {ok_wid}")
assert ok_len and ok_wid, "recovered scale puts the pitch outside the regulation envelope"
RESULTS["pitch_dims_m"] = {"length": float(pl), "width": float(pw)}

## 14. Cross-check 2 — the athletics track

Dasharath Rangasala's football pitch sits inside a 400 m athletics track. A standard 8-lane track
has an outer edge of roughly 176.9 m x 92.5 m. This is a weaker control than the pitch markings —
the outer edge is a soft boundary against the stands and the morphological closing used to isolate
the track surface moves it by a pixel or two — so it is reported as corroboration, not evidence.

What it is good for is the same thing as section 13: it would catch a factor-of-two error instantly.

In [ ]:
red = (((Hc < 14) | (Hc > 168)) & (Sc > 60) & (Vc > 60)).astype(np.uint8)
red = cv2.morphologyEx(red, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
red = cv2.morphologyEx(red, cv2.MORPH_OPEN, np.ones((7, 7), np.uint8))
nn, ll, ss, cc = cv2.connectedComponentsWithStats(red, 8)
near = [i for i in range(1, nn) if ss[i, 4] > 20000 and
        np.hypot(cc[i][0] - cx, cc[i][1] - cy) < 400]
if near:
    k = max(near, key=lambda i: ss[i, 4])
    c = max(cv2.findContours((ll == k).astype(np.uint8), cv2.RETR_EXTERNAL,
                             cv2.CHAIN_APPROX_NONE)[0], key=cv2.contourArea)
    (_, _), (ta, tb), _ = cv2.minAreaRect(c)
    tl, tw = max(ta, tb), min(ta, tb)
    print(f"track outer rect : {tl:.1f} x {tw:.1f} px -> {tl*GSD:.1f} x {tw*GSD:.1f} m")
    print(f"standard 8-lane  : {cfg.TRACK_OUTER_LEN:.1f} x {cfg.TRACK_OUTER_WID:.1f} m")
    print(f"implied GSD from track length: {cfg.TRACK_OUTER_LEN/tl:.4f} m/px "
          f"({(cfg.TRACK_OUTER_LEN/tl - GSD)/GSD*100:+.1f}% vs the pitch estimate)")
    RESULTS["track_check"] = {"len_px": float(tl), "wid_px": float(tw),
                              "implied_gsd": float(cfg.TRACK_OUTER_LEN / tl)}
else:
    print("track surface not isolated — cross-check skipped")

## 15. Cross-check 3 — a 50 m pool in a different tile

The strongest kind of corroboration is a control object in a **different tile**, because it tests the
assumption the whole dataset rests on: that every tile was exported at one fixed scale. If tile 1 and
tile 264 disagree, the Atlas scale changed mid-export and no single GSD exists.

`image1.png` contains a swimming pool. Long-course competition pools are 50 m; the measured aspect
ratio distinguishes the case, since a 50 m pool with eight 2.5 m lanes has an aspect near 2.4-2.5
while a 25 m pool at 12.5 m wide would be 2.0.

This is also a correction of the record. An earlier estimate treated this pool as 25 m and arrived at
half the true GSD. The aspect ratio is what settles it, and the pitch measurement is what confirms
which reading is right.

In [ ]:
pool_path = IMG_DIR / cfg.POOL_TILE
if pool_path.exists():
    pim = cv2.imread(str(pool_path), cv2.IMREAD_COLOR)
    ph = cv2.cvtColor(pim, cv2.COLOR_BGR2HSV)
    pm = ((ph[..., 0] > 80) & (ph[..., 0] < 105) &
          (ph[..., 1] > 90) & (ph[..., 2] > 90)).astype(np.uint8)
    pm = cv2.morphologyEx(pm, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    cs, _ = cv2.findContours(pm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cs:
        c = max(cs, key=cv2.contourArea)
        (_, _), (a, b), _ = cv2.minAreaRect(c)
        pl_px, pw_px = max(a, b), min(a, b)
        print(f"pool water surface : {pl_px:.1f} x {pw_px:.1f} px, aspect {pl_px/pw_px:.2f}")
        print(f"  at the recovered GSD -> {pl_px*GSD:.1f} x {pw_px*GSD:.1f} m")
        print(f"  50 m hypothesis -> GSD {cfg.POOL_LEN/pl_px:.4f} m/px "
              f"({(cfg.POOL_LEN/pl_px - GSD)/GSD*100:+.1f}% vs the pitch estimate)")
        print(f"  25 m hypothesis -> GSD {25.0/pl_px:.4f} m/px "
              f"({(25.0/pl_px - GSD)/GSD*100:+.1f}% vs the pitch estimate) <- rejected")
        RESULTS["pool_check"] = {"len_px": float(pl_px), "wid_px": float(pw_px),
                                 "aspect": float(pl_px / pw_px),
                                 "implied_gsd_50m": float(cfg.POOL_LEN / pl_px)}
else:
    print(f"{cfg.POOL_TILE} not found — cross-check skipped")

## 16. Rounding to the export configuration

A QGIS layout export does not produce an arbitrary scale. It produces `ground per pixel = (scale) x
(25.4 / dpi) / 1000` metres, so any real export lands on the value implied by whatever round scale
and round DPI were set a year ago. Finding a plausible pair that reproduces the measurement is not
proof, but a measurement that lands on a configuration nobody would ever type should raise doubts,
and one that lands on a common pair supports the result.

The candidates below are enumerated over common DPI values and round scales, and the closest are
printed. The paper should quote the measured value with its uncertainty and may note the likely
configuration as an aside — not the other way round.

In [ ]:
cands = []
for dpi in (72, 96, 150, 200, 254, 300, 400, 600):
    for scale in list(range(1000, 6001, 100)):
        g = scale * (25.4 / dpi) / 1000.0
        if abs(g - GSD) / GSD < 0.05:
            cands.append({"dpi": dpi, "scale": f"1:{scale}", "gsd": g,
                          "dev_pct": (g - GSD) / GSD * 100,
                          "tile_side_m": g * cfg.NAT_W})
cd = pd.DataFrame(cands).reindex(pd.DataFrame(cands).dev_pct.abs().sort_values().index)
print(cd.head(10).to_string(index=False, float_format=lambda v: f"{v:10.4f}"))

GSD_ROUND = 0.30
print(f"\nmeasured  : {GSD:.4f} m/px  -> tile {GSD*cfg.NAT_W:.1f} x {GSD*cfg.NAT_H:.1f} m")
print(f"adopted   : {GSD_ROUND:.4f} m/px  -> tile {GSD_ROUND*cfg.NAT_W:.1f} x "
      f"{GSD_ROUND*cfg.NAT_H:.1f} m   (deviation {(GSD_ROUND-GSD)/GSD*100:+.2f}%)")
RESULTS["configuration_candidates"] = cd.head(10).to_dict("records")
RESULTS["gsd_adopted"] = GSD_ROUND

## 17. Consequences for the dataset description

Every metre-denominated statistic in the characterisation notebook was computed under the
unverified 300 m assumption. All of them scale analytically, so nothing has to be recomputed from
the imagery:

- lengths and widths multiply by `k = true_side / 300`
- areas multiply by `k^2`
- densities per unit area divide by `k`

Everything expressed in pixels, and every model result, is unaffected — the models were trained on
512 px rasters and never saw a metre. What changes is only how the dataset is *described*.

The one substantive consequence is framing. At the recovered scale the working resolution is about
1.37 m/px, not the 0.59 m/px previously believed. That is ordinary very-high-resolution satellite
territory, not sub-metre, and any claim to sub-metre resolution has to come out of the paper. It
also explains the thin-class difficulty more honestly: a road at 11.5 m median width is 8.4 px at
512 and about 2 px at a stride-4 decoder output.

In [ ]:
TILE_SIDE = GSD_ROUND * cfg.NAT_W
k = TILE_SIDE / cfg.OLD_TILE_SIDE_M
GSD_WORK = TILE_SIDE / cfg.IMG_SIZE
N_TILES = 408
tile_ha = (GSD_ROUND * cfg.NAT_W) * (GSD_ROUND * cfg.NAT_H) / 1e4
annot_km2 = N_TILES * tile_ha / 100

print(f"correction factor k = {k:.4f}  (areas x {k**2:.4f}, densities / {k:.4f})\n")

# measured under the 300 m assumption (characterisation notebook, this dataset)
OLD = {
    "native GSD (m/px)":            (0.1279, 1),
    "working GSD at 512 (m/px)":    (0.5859, 1),
    "tile side (m)":                (300.0, 1),
    "tile area (ha)":               (9.00, 2),
    "annotated coverage (km2)":     (36.72, 2),
    "Road median width (m)":        (4.92, 1),
    "Road p10 width (m)":           (2.57, 1),
    "Road p90 width (m)":           (11.48, 1),
    "Road total length (km)":       (84.5, 1),
    "Road density (km/km2)":        (2.30, -1),
    "River median width (m)":       (6.32, 1),
    "River p10 width (m)":          (3.52, 1),
    "River p90 width (m)":          (15.47, 1),
    "River total length (km)":      (24.0, 1),
    "River density (km/km2)":       (0.65, -1),
    "edge density (m/ha)":          (189.2, -1),
    "Residential mean patch (m2)":  (30850.87, 2),
    "Road mean patch (m2)":         (2164.74, 2),
    "River mean patch (m2)":        (2273.25, 2),
    "Forest mean patch (m2)":       (24870.14, 2),
    "UnusedLand mean patch (m2)":   (16077.18, 2),
    "Agricultural mean patch (m2)": (25328.63, 2),
}
rows = []
for name, (val, p) in OLD.items():
    if name == "native GSD (m/px)":
        new = GSD_ROUND
    elif name == "working GSD at 512 (m/px)":
        new = GSD_WORK
    elif name == "tile side (m)":
        new = TILE_SIDE
    elif name == "tile area (ha)":
        new = tile_ha
    elif name == "annotated coverage (km2)":
        new = annot_km2
    else:
        new = val * (k ** p) if p > 0 else val / k
    rows.append({"statistic": name, "assumed_300m": val, "corrected": new,
                 "factor": new / val})
resc = pd.DataFrame(rows)
print(resc.to_string(index=False, float_format=lambda v: f"{v:14.4f}"))

SHARES = {"Residential": 56.020, "Road": 1.569, "River": 0.595, "Forest": 10.385,
          "UnusedLand": 12.967, "Agricultural": 18.100, "<unlabelled>": 0.365}
print("\nclass areas at the corrected scale:")
areas = []
for nm, s in SHARES.items():
    a = annot_km2 * s / 100
    areas.append({"class": nm, "share_pct": s, "area_km2": a, "area_ha": a * 100})
    print(f"  {nm:<14}{s:6.2f}%  {a:8.2f} km2")
adf = pd.DataFrame(areas)

resc.to_csv(f"{cfg.OUT_DIR}/rescaled_statistics.csv", index=False)
adf.to_csv(f"{cfg.OUT_DIR}/class_areas_corrected.csv", index=False)
RESULTS["rescale"] = {"k": k, "tile_side_m": TILE_SIDE, "gsd_work": GSD_WORK,
                      "tile_area_ha": tile_ha, "annotated_km2": annot_km2,
                      "table": resc.to_dict("records"), "class_areas": areas}

## 18. Figures

Four figures at 300 dpi, PNG and vector PDF. The first two are the evidence a reader needs to accept
the measurement without rerunning it: where the pitch was found, and where each line was detected.

In [ ]:
def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{cfg.FIG_DIR}/{name}.{ext}", bbox_inches="tight")
    plt.close(fig)
    print("  wrote", name)

# 1 -- the located pitch in the whole tile
box = cv2.boxPoints(((cx, cy), (L, W), ANG)).astype(int)
vis = img.copy()
cv2.drawContours(vis, [box], 0, (0, 0, 255), 6)
fig, a = plt.subplots(figsize=(6, 6))
a.imshow(vis[..., ::-1]); a.set_xticks([]); a.set_yticks([])
a.set_title(f"{cfg.CALIB_TILE} — located pitch ({L:.0f} x {W:.0f} px)")
save(fig, "fig_pitch_located")

# 2 -- the rotated pitch with every detected line
fig, a = plt.subplots(figsize=(8, 5.6))
a.imshow(P[..., ::-1])
for x, lbl in [(gl_L, "goal"), (ga_L, "goal area"), (pa_L, "penalty"), (half, "halfway"),
               (pa_R, "penalty"), (ga_R, "goal area"), (gl_R, "goal")]:
    a.axvline(x, color="yellow", lw=0.9)
    a.text(x, -6, lbl, rotation=90, fontsize=6, color="k", ha="center", va="bottom")
for y in (touch[0], touch[1], CENTRE_Y):
    a.axhline(y, color="cyan", lw=0.7, ls=":")
a.set_xticks([]); a.set_yticks([]); a.set_title("detected markings on the rotated pitch")
save(fig, "fig_pitch_lines")

# 3 -- the profiles
fig, ax = plt.subplots(2, 1, figsize=(8, 4.4))
ax[0].plot(col_prof, lw=0.9); ax[0].set_title("longitudinal profile (transverse lines)")
for x in (gl_L, ga_L, pa_L, half, pa_R, ga_R, gl_R):
    ax[0].axvline(x, color="r", lw=0.7, ls="--")
pr = white[:, int(gl_L + 4):int(pa_L - 3)].mean(1)
ax[1].plot(pr, lw=0.9); ax[1].set_title("transverse profile inside the left penalty area")
for y in (touch[0], touch[1]):
    ax[1].axvline(y, color="c", lw=0.7, ls=":")
for y in found["left penalty"]:
    pass
ax[1].axvline(CENTRE_Y, color="k", lw=0.6)
for a_ in ax:
    a_.set_ylabel("white excess")
fig.tight_layout()
save(fig, "fig_profiles")

# 4 -- the four estimates
fig, a = plt.subplots(figsize=(6, 3.2))
a.errorbar(range(len(est)), est.gsd_m_per_px, fmt="o", ms=7, color="tab:blue")
a.axhline(GSD, color="tab:red", lw=1.2, label=f"weighted {GSD:.4f}")
a.axhline(GSD_ROUND, color="k", lw=1.0, ls="--", label=f"adopted {GSD_ROUND:.2f}")
a.set_xticks(range(len(est)))
a.set_xticklabels(est.control, rotation=25, ha="right", fontsize=7)
a.set_ylabel("GSD (m/px)"); a.legend(fontsize=7, frameon=False)
save(fig, "fig_gsd_estimates")

## 19. Export and the citable summary

Everything measured goes to `gsd_calibration/`. The LaTeX file contains the control table as a
`booktabs` body, so the calibration table in the appendix can be pasted rather than retyped.

In [ ]:
def _j(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

RESULTS["_meta"] = {"notebook": "geonusaf-gsd-calibration",
                    "cfg": {k_: (list(v) if isinstance(v, tuple) else v)
                            for k_, v in asdict(cfg).items()}}
json.dump(RESULTS, open(f"{cfg.OUT_DIR}/gsd_calibration.json", "w"), indent=2, default=_j)
est.to_csv(f"{cfg.OUT_DIR}/gsd_controls.csv", index=False)

LATEX["gsd_controls"] = "\n".join(
    f"{r.control} & {r.true_m:.2f} & {r.px:.2f} & {r.gsd_m_per_px:.4f} \\\\"
    for r in est.itertuples())
LATEX["rescale"] = "\n".join(
    f"{r.statistic} & {r.assumed_300m:.2f} & {r.corrected:.2f} \\\\"
    for r in resc.itertuples())
with open(f"{cfg.OUT_DIR}/latex_tables.tex", "w") as fh:
    for k_, v in LATEX.items():
        fh.write(f"% ---- {k_} ----\n{v}\n\n")

print("written to", cfg.OUT_DIR)
for f in sorted(os.listdir(cfg.OUT_DIR)): print("  ", f)

print("\n" + "=" * 78)
print("CALIBRATION SUMMARY")
print("=" * 78)
print(f"calibration tile   : {cfg.CALIB_TILE} ({cfg.NAT_W} x {cfg.NAT_H} px)")
print(f"control objects    : 4 fixed dimensions from IFAB Law 1")
print(f"symmetry residual  : {err_half:.2f} px on a {pitch_len_px:.0f} px baseline")
print(f"measured GSD       : {GSD:.4f} +/- {half_range:.4f} m/px ({rel:.1f}%)")
print(f"adopted GSD        : {GSD_ROUND:.3f} m/px")
print(f"tile footprint     : {GSD_ROUND*cfg.NAT_W:.1f} m x {GSD_ROUND*cfg.NAT_H:.1f} m "
      f"= {tile_ha:.2f} ha")
print(f"working GSD at 512 : {GSD_WORK:.3f} m/px")
print(f"annotated coverage : {N_TILES} tiles = {annot_km2:.1f} km2")
print(f"correction factor  : lengths x {k:.3f}, areas x {k**2:.3f}, densities / {k:.3f}")
print(f"pitch validation   : {pl:.1f} m x {pw:.1f} m, inside the Law 1 envelope")
print("=" * 78)

## 20. Run checklist

1. Attach `sugam24/satellite-datasets`. No GPU, no internet.
2. Run top to bottom. Total runtime is under a minute.
3. Three asserts carry the result. If any fires, the number must not be used:
   - section 3: the tile is 2345 x 2351 px;
   - section 9: the halfway line sits within 3 px of the goal-line midpoint;
   - section 12: no control disagrees with the combined estimate by more than 3 px.
4. `Save Version`, download `gsd_calibration/`. `gsd_calibration.json` holds every number,
   `rescaled_statistics.csv` is the corrected dataset table, `figs/*.pdf` are the figures.
5. Set `TILE_SIDE_M = 703.5` in the characterisation notebook and re-run it; the cached npz makes
   that a two-minute job and it regenerates every metre-denominated table consistently.

### How to state this in the paper

The scale is a *recovered* quantity, not a recorded one, and the paper should say so plainly. A
defensible form: the tile footprint was recovered photogrammetrically from four fixed-dimension
features of a regulation football pitch present in one tile, giving 0.30 m/px (703.5 m x 705.3 m per
tile) with a 1-2 % uncertainty; the original Atlas export configuration was not retained. Put the
control table in the appendix. Reviewers accept a stated uncertainty on a recovered parameter; they
do not accept a confident number with nothing behind it.

### What this does not recover

Tile positions. Scale is one number shared by all 408 tiles; position is 408 pairs of coordinates,
and no amount of measurement inside a tile can produce them. Spatial cross-validation therefore still
rests on the export-order proxy, and DEM conditioning is still impossible.